In [1]:

import tkinter as tk
from tkinter import messagebox
import cv2
from PIL import Image, ImageTk


class QRCodeReaderApp:

    def __init__(self, root):

        self.root = root
        self.root.title("QR Code Camera Reader")
        self.root.geometry("1100x850")

        self.camera = None
        self.running = False
        self.photo = None

        # Create QR detector
        self.qr_detector = cv2.QRCodeDetector()

        # --------------------------------
        # TITLE
        # --------------------------------
        title = tk.Label(
            root,
            text="QR Code Camera Reader",
            font=("Arial", 22, "bold")
        )
        title.pack(pady=10)

        # --------------------------------
        # CAMERA DISPLAY
        # --------------------------------
        self.camera_label = tk.Label(
            root,
            bg="black",
            fg="white",
            text="Camera is not running"
        )
        self.camera_label.pack(pady=10)

        # --------------------------------
        # BUTTONS
        # --------------------------------
        button_frame = tk.Frame(root)
        button_frame.pack(pady=10)

        self.start_button = tk.Button(
            button_frame,
            text="Start Camera",
            width=15,
            font=("Arial", 12, "bold"),
            command=self.start_camera
        )
        self.start_button.grid(row=0, column=0, padx=10)

        self.stop_button = tk.Button(
            button_frame,
            text="Stop Camera",
            width=15,
            font=("Arial", 12, "bold"),
            command=self.stop_camera
        )
        self.stop_button.grid(row=0, column=1, padx=10)

        # --------------------------------
        # QR RESULT
        # --------------------------------
        result_title = tk.Label(
            root,
            text="QR Code Result:",
            font=("Arial", 14, "bold")
        )
        result_title.pack(pady=5)

        self.result_text = tk.Text(
            root,
            height=4,
            width=80,
            font=("Arial", 12)
        )
        self.result_text.pack(pady=5)

        # --------------------------------
        # COPY BUTTON
        # --------------------------------
        copy_button = tk.Button(
            root,
            text="Copy Result",
            width=15,
            command=self.copy_result
        )
        copy_button.pack(pady=5)

        # --------------------------------
        # STATUS
        # --------------------------------
        self.status_label = tk.Label(
            root,
            text="Ready",
            font=("Arial", 11)
        )
        self.status_label.pack(pady=10)

        # Handle closing window
        self.root.protocol(
            "WM_DELETE_WINDOW",
            self.close_application
        )

    # ====================================
    # START CAMERA
    # ====================================

    def start_camera(self):

        if self.running:
            return

        self.camera = cv2.VideoCapture(0)

        if not self.camera.isOpened():

            self.camera.release()
            self.camera = None

            messagebox.showerror(
                "Camera Error",
                "Could not open camera."
            )
            return

        # Request higher camera resolution
        self.camera.set(
            cv2.CAP_PROP_FRAME_WIDTH, 1280
        )

        self.camera.set(
            cv2.CAP_PROP_FRAME_HEIGHT, 720
        )

        # Enable autofocus if supported
        self.camera.set(
            cv2.CAP_PROP_AUTOFOCUS, 1
        )

        self.running = True

        self.status_label.config(
            text="Scanning for QR codes..."
        )

        self.update_camera()

    # ====================================
    # DETECT QR CODES
    # ====================================

    def detect_qr(self, frame):

        # Convert frame to grayscale
        gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        # --------------------------------
        # TRY MULTIPLE QR CODE DETECTION
        # --------------------------------

        try:

            found, decoded_info, points, _ = (
                self.qr_detector.detectAndDecodeMulti(gray)
            )

            if found and points is not None:

                results = []

                for i, text in enumerate(decoded_info):

                    if text:

                        results.append(text)

                        # Draw boundary around QR code
                        polygon = points[i].astype(int)

                        cv2.polylines(
                            frame,
                            [polygon],
                            True,
                            (0, 255, 0),
                            4
                        )

                if results:

                    return results

        except cv2.error:

            pass

        # --------------------------------
        # FALLBACK: SINGLE QR CODE
        # --------------------------------

        text, points, _ = self.qr_detector.detectAndDecode(gray)

        if text:

            if points is not None:

                polygon = points.astype(int)

                cv2.polylines(
                    frame,
                    [polygon],
                    True,
                    (0, 255, 0),
                    4
                )

            return [text]

        return []

    # ====================================
    # UPDATE CAMERA
    # ====================================

    def update_camera(self):

        if not self.running:
            return

        success, frame = self.camera.read()

        if not success:

            self.status_label.config(
                text="Failed to read camera."
            )

            self.stop_camera()
            return

        # --------------------------------
        # DETECT QR CODE
        # --------------------------------

        results = self.detect_qr(frame)

        if results:

            # Display detected QR codes
            result_string = "\n".join(results)

            self.result_text.delete(
                "1.0",
                tk.END
            )

            self.result_text.insert(
                tk.END,
                result_string
            )

            self.status_label.config(
                text="QR Code detected!"
            )

        else:

            self.status_label.config(
                text="Scanning... Point camera at QR code."
            )

        # --------------------------------
        # DISPLAY CAMERA
        # --------------------------------

        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        image = Image.fromarray(frame_rgb)

        # Large display
        image = image.resize(
            (900, 600)
        )

        self.photo = ImageTk.PhotoImage(image)

        self.camera_label.config(
            image=self.photo,
            text=""
        )

        # Schedule next frame
        self.root.after(
            10,
            self.update_camera
        )

    # ====================================
    # STOP CAMERA
    # ====================================

    def stop_camera(self):

        self.running = False

        if self.camera is not None:

            self.camera.release()
            self.camera = None

        self.camera_label.config(
            image="",
            text="Camera is not running",
            bg="black",
            fg="white"
        )

        self.photo = None

        self.status_label.config(
            text="Camera stopped."
        )

    # ====================================
    # COPY RESULT
    # ====================================

    def copy_result(self):

        text = self.result_text.get(
            "1.0",
            tk.END
        ).strip()

        if not text:

            messagebox.showwarning(
                "Empty Result",
                "No QR code has been detected."
            )
            return

        self.root.clipboard_clear()

        self.root.clipboard_append(text)

        messagebox.showinfo(
            "Copied",
            "QR code result copied."
        )

    # ====================================
    # CLOSE APPLICATION
    # ====================================

    def close_application(self):

        self.stop_camera()

        self.root.destroy()


# ========================================
# MAIN
# ========================================

if __name__ == "__main__":

    root = tk.Tk()

    app = QRCodeReaderApp(root)

    root.mainloop()
